# 전체 정류소 모델 성능 요약
`run_all_stations.py` 실행 결과(`reports/results_all.csv`)를 읽어 분석합니다.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

sys.path.insert(0, str(Path("..").resolve()))
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.family"] = "DejaVu Sans"

RESULTS_CSV = Path("..") / "reports" / "results_all.csv"
assert RESULTS_CSV.exists(), f"결과 파일 없음: {RESULTS_CSV}"

## 1. 데이터 로드

In [ ]:
df = pd.read_csv(RESULTS_CSV)
print(f"총 {len(df):,}행  |  정류소 {df['station_id'].nunique()}개  |  모델 {df['model'].nunique()}개")
print(f"모델: {df['model'].unique().tolist()}")
df.head()

## 2. 모델별 전체 성능 (fold 평균)

In [ ]:
summary = (
    df.groupby(["model", "day_type"])[["RMSE", "MAE", "Asym_RMSE", "Under_pred_rate"]]
    .mean()
    .round(3)
)
print(summary.to_string())

## 3. 모델 비교 시각화

In [ ]:
for day_type in ["weekday", "weekend"]:
    sub = df[df["day_type"] == day_type]
    if sub.empty:
        continue

    model_avg = sub.groupby("model")[["RMSE", "Asym_RMSE", "Under_pred_rate"]].mean().round(3)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(f"Model Comparison — {day_type}", fontsize=13, fontweight="bold")

    model_avg["RMSE"].plot(kind="bar", ax=axes[0], color="steelblue", rot=20)
    axes[0].set_title("RMSE (lower = better)")
    axes[0].yaxis.set_major_formatter(ticker.FormatStrFormatter("%.3f"))

    model_avg["Asym_RMSE"].plot(kind="bar", ax=axes[1], color="darkorange", rot=20)
    axes[1].set_title("Asymmetric RMSE α=2 (main metric)")
    axes[1].yaxis.set_major_formatter(ticker.FormatStrFormatter("%.3f"))

    model_avg["Under_pred_rate"].plot(kind="bar", ax=axes[2], color="tomato", rot=20)
    axes[2].set_title("Under-prediction rate (%)")

    plt.tight_layout()
    plt.show()

## 4. 정류소별 Asym_RMSE 히트맵

In [ ]:
for day_type in ["weekday", "weekend"]:
    sub = df[df["day_type"] == day_type]
    if sub.empty:
        continue

    pivot = (
        sub.groupby(["station_id", "model"])["Asym_RMSE"]
        .mean()
        .unstack("model")
        .round(3)
    )

    fig, ax = plt.subplots(figsize=(len(pivot.columns) * 2, len(pivot) * 0.4 + 1))
    sns.heatmap(
        pivot, annot=True, fmt=".3f", cmap="YlOrRd",
        linewidths=0.5, ax=ax
    )
    ax.set_title(f"Asym_RMSE per Station — {day_type}", fontsize=12)
    ax.set_xlabel("")
    ax.set_ylabel("Station ID")
    plt.tight_layout()
    plt.show()

## 5. 정류소별 최선 모델

In [ ]:
station_model_avg = (
    df.groupby(["station_id", "day_type", "model"])["Asym_RMSE"]
    .mean()
    .reset_index()
)

best = (
    station_model_avg
    .loc[station_model_avg.groupby(["station_id", "day_type"])["Asym_RMSE"].idxmin()]
    .sort_values(["day_type", "station_id"])
    .reset_index(drop=True)
)

print("정류소별 Asym_RMSE 최소 모델:")
print(best.to_string(index=False))

## 6. Rolling fold별 안정성 확인

In [ ]:
fold_avg = (
    df.groupby(["model", "day_type", "fold"])["Asym_RMSE"]
    .mean()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 4))
for (model, day_type), grp in fold_avg.groupby(["model", "day_type"]):
    ax.plot(grp["fold"], grp["Asym_RMSE"], marker="o", label=f"{model} ({day_type})")

ax.set_xlabel("Fold (1=최근, n=과거)")
ax.set_ylabel("Asym_RMSE (평균)")
ax.set_title("Fold별 성능 변화 — 시간적 안정성")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()